<a href="https://colab.research.google.com/github/Nilufar-Komil/Applications-of-the-Generalized-Stiefel-Manifold-in-NN/blob/small_run/ViT_stiefel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# -*- coding: utf-8 -*-

import os
import sys
import math
import random
from typing import Optional

sys.path.append(os.getcwd())

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.optim import SGD
from torch.utils.data import Dataset, DataLoader, random_split

# Data processing and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device available:", device)


Device available: cpu


In [2]:

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split

# ------------------------------------------------------------
# CIFAR-10 normalization (same as vit_sgd)
# ------------------------------------------------------------

cifar10_mean = (125.3 / 255.0, 123.0 / 255.0, 113.9 / 255.0)
cifar10_std  = (63.0 / 255.0, 62.1 / 255.0, 66.7 / 255.0)

# ------------------------------------------------------------
# transforms
# ------------------------------------------------------------

# Training transform:
# random horizontal flip + reflection padding
# + random crop + normalization

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Pad(4, padding_mode='reflect'),
    transforms.RandomCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# Validation/test transform:
# no random augmentation, only normalization

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# ------------------------------------------------------------
# create two versions of CIFAR-10 train data
# ------------------------------------------------------------

# Augmented version for training
full_train_dataset_aug = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform,
)

# Clean version for validation
full_train_dataset_clean = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=test_transform,
)

# Test dataset: clean transform only
test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform,
)

# ------------------------------------------------------------
# split full training dataset into train and validation
# ------------------------------------------------------------

train_size = int(0.8 * len(full_train_dataset_aug))
val_size = len(full_train_dataset_aug) - train_size

# Same seed as vit_sgd
split_seed = 42

train_dataset, _ = random_split(
    full_train_dataset_aug,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(split_seed),
)

_, val_dataset = random_split(
    full_train_dataset_clean,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(split_seed),
)

# ------------------------------------------------------------
# print dataset sizes
# ------------------------------------------------------------

print("Full train size:", len(full_train_dataset_aug))
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))


100%|██████████| 170M/170M [09:04<00:00, 313kB/s]


Full train size: 50000
Train size: 40000
Val size: 10000
Test size: 10000


In [3]:

# ------------------------------------------------------------
# Batch size
# ------------------------------------------------------------

batch_size = 128

# ------------------------------------------------------------
# Random seeds
# ------------------------------------------------------------

seed = 42

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ------------------------------------------------------------
# cuDNN settings
# ------------------------------------------------------------

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = False


In [4]:

# ------------------------------------------------------------
# ViT-Tiny/4: CIFAR-10 DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [5]:

# ------------------------------------------------------------
# Multi-Head Self-Attention
# ------------------------------------------------------------

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=192, num_heads=3,
                 dropout=0.0):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Separate per-head Q, K, V matrices
        # Shape: (num_heads, embed_dim, head_dim)

        shape = (num_heads, embed_dim, self.head_dim)

        self.W_q = nn.Parameter(torch.empty(shape))
        self.W_k = nn.Parameter(torch.empty(shape))
        self.W_v = nn.Parameter(torch.empty(shape))

        self.b_q = nn.Parameter(
            torch.zeros(num_heads, self.head_dim)
        )
        self.b_k = nn.Parameter(
            torch.zeros(num_heads, self.head_dim)
        )
        self.b_v = nn.Parameter(
            torch.zeros(num_heads, self.head_dim)
        )

        self.proj = nn.Linear(embed_dim, embed_dim)

        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

        self.reset_parameters()

    def reset_parameters(self):
        # Orthogonal initialization
        for h in range(self.num_heads):
            nn.init.orthogonal_(self.W_q[h])
            nn.init.orthogonal_(self.W_k[h])
            nn.init.orthogonal_(self.W_v[h])

        nn.init.zeros_(self.b_q)
        nn.init.zeros_(self.b_k)
        nn.init.zeros_(self.b_v)

        nn.init.orthogonal_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def project(self, x, W, bias):
        # x: (batch, tokens, embed_dim)
        # W: (heads, embed_dim, head_dim)

        out = torch.einsum("bnd,hdp->bhnp", x, W)

        return out + bias[None, :, None, :]

    def forward(self, x):
        B, N, C = x.shape

        q = self.project(x, self.W_q, self.b_q)
        k = self.project(x, self.W_k, self.b_k)
        v = self.project(x, self.W_v, self.b_v)

        # Attention scores
        attn = (q @ k.transpose(-2, -1)) * self.scale

        attn = torch.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        # Weighted sum of values
        out = attn @ v

        # Combine attention heads
        out = out.transpose(1, 2).reshape(B, N, C)

        out = self.proj(out)
        out = self.proj_drop(out)

        return out


# ------------------------------------------------------------
# Transformer MLP
# ------------------------------------------------------------

class TransformerMLP(nn.Module):
    def __init__(self, embed_dim=192, mlp_ratio=4.0,
                 dropout=0.0):
        super().__init__()

        hidden_dim = int(embed_dim * mlp_ratio)

        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(dropout)

        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.drop2 = nn.Dropout(dropout)

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.orthogonal_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)

        nn.init.orthogonal_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)

        x = self.fc2(x)
        x = self.drop2(x)

        return x


# ------------------------------------------------------------
# ViT Transformer Block
# Replaces WRNBasicBlock
# ------------------------------------------------------------

class ViTBasicBlock(nn.Module):
    def __init__(self, embed_dim=192, num_heads=3,
                 mlp_ratio=4.0, dropout=0.0):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)

        self.attn = MultiHeadSelfAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
        )

        self.norm2 = nn.LayerNorm(embed_dim)

        self.mlp = TransformerMLP(
            embed_dim=embed_dim,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.ones_(self.norm1.weight)
        nn.init.zeros_(self.norm1.bias)

        nn.init.ones_(self.norm2.weight)
        nn.init.zeros_(self.norm2.bias)

    def forward(self, x):
        # Multi-head attention + residual
        x = x + self.attn(self.norm1(x))

        # MLP + residual
        x = x + self.mlp(self.norm2(x))

        return x


In [6]:

import torch
import torch.nn as nn


# ------------------------------------------------------------
# ViT Group: replaces WRNGroup
# ------------------------------------------------------------

class ViTGroup(nn.Module):
    def __init__(self, num_blocks, embed_dim=192,
                 num_heads=3, mlp_ratio=4.0, dropout=0.0):
        super().__init__()

        blocks = []

        for i in range(num_blocks):
            blocks.append(
                ViTBasicBlock(
                    embed_dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    dropout=dropout,
                )
            )

        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        return self.blocks(x)


# ------------------------------------------------------------
# ViT-Tiny/4: replaces WideResNet
# ------------------------------------------------------------

class ViTTiny4(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=4,
        depth=12,
        embed_dim=192,
        num_heads=3,
        num_classes=10,
        mlp_ratio=4.0,
        dropout=0.0,
    ):
        super().__init__()

        assert img_size % patch_size == 0
        assert depth % 3 == 0
        assert embed_dim % num_heads == 0

        self.img_size = img_size
        self.patch_size = patch_size

        self.num_patches = (img_size // patch_size) ** 2

        # ----------------------------------------------------
        # Patch embedding
        # (B, 3, 32, 32) -> (B, 192, 8, 8)
        # ----------------------------------------------------

        self.patch_embed = nn.Conv2d(
            in_channels=3,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            bias=True,
        )

        # Classification token
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, embed_dim)
        )

        # Learnable position embeddings
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches + 1, embed_dim)
        )

        self.pos_drop = nn.Dropout(dropout)

        # ----------------------------------------------------
        # Transformer encoder groups
        # 12 blocks = 3 groups x 4 blocks
        # ----------------------------------------------------

        blocks_per_group = depth // 3

        self.group0 = ViTGroup(
            num_blocks=blocks_per_group,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

        self.group1 = ViTGroup(
            num_blocks=blocks_per_group,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

        self.group2 = ViTGroup(
            num_blocks=blocks_per_group,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

        # Final normalization
        self.norm = nn.LayerNorm(embed_dim)

        # Classification head
        self.head = nn.Linear(embed_dim, num_classes)

        self.reset_parameters()

    # --------------------------------------------------------
    # Initialization
    # --------------------------------------------------------

    def reset_parameters(self):
        nn.init.orthogonal_(self.patch_embed.weight)
        nn.init.zeros_(self.patch_embed.bias)

        nn.init.normal_(
            self.cls_token, mean=0.0, std=0.02
        )

        nn.init.normal_(
            self.pos_embed, mean=0.0, std=0.02
        )

        nn.init.ones_(self.norm.weight)
        nn.init.zeros_(self.norm.bias)

        nn.init.normal_(
            self.head.weight,
            mean=0.0,
            std=2 / (self.head.in_features ** 0.5)
        )
        nn.init.zeros_(self.head.bias)

    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    def forward(self, x):

        # Patch embedding
        # (B, 3, 32, 32) -> (B, 192, 8, 8)
        x = self.patch_embed(x)

        # Flatten spatial dimensions
        # (B, 192, 8, 8) -> (B, 64, 192)
        x = x.flatten(2).transpose(1, 2)

        # Add classification token
        cls_token = self.cls_token.expand(
            x.shape[0], -1, -1
        )

        x = torch.cat((cls_token, x), dim=1)

        # Add positional embeddings
        # (B, 65, 192)
        x = self.pos_drop(x + self.pos_embed)

        # Transformer groups
        x = self.group0(x)
        x = self.group1(x)
        x = self.group2(x)

        # Final LayerNorm
        x = self.norm(x)

        # Extract CLS token
        x = x[:, 0]

        # Classification
        x = self.head(x)

        return x


# ------------------------------------------------------------
# Initialize ViT-Tiny/4
# ------------------------------------------------------------

model = ViTTiny4(
    img_size=32,
    patch_size=4,
    depth=12,
    embed_dim=192,
    num_heads=3,
    num_classes=10,
    mlp_ratio=4.0,
    dropout=0.0,
).to(device)

# ------------------------------------------------------------
# Loss function
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()

print("Total parameters:",
      sum(p.numel() for p in model.parameters()))


Total parameters: 5362762


In [7]:

# ------------------------------------------------------------
# Split parameters into Stiefel and normal parameters
# ------------------------------------------------------------

stiefel_params = []
normal_params = []

stiefel_param_ids = set()

for module_name, module in model.named_modules():

    # Select Q and K matrices from attention layers
    if isinstance(module, MultiHeadSelfAttention):

        for weight_name in ["W_q", "W_k"]:

            weight = getattr(module, weight_name)

            if not weight.requires_grad:
                continue

            # Shape: (num_heads, embed_dim, head_dim)
            num_heads, n, p = weight.shape

            # Feasibility for St(n,p): n >= p
            if n >= p:

                stiefel_params.append(weight)
                stiefel_param_ids.add(id(weight))

                print(
                    f"Stiefel layer: "
                    f"{module_name}.{weight_name}, "
                    f"heads={num_heads}, "
                    f"matrix shape=({n}, {p})"
                )

            else:

                print(
                    f"Normal layer, not feasible for Stiefel: "
                    f"{module_name}.{weight_name}, "
                    f"shape=({n}, {p})"
                )

# ------------------------------------------------------------
# Collect remaining Euclidean parameters
# ------------------------------------------------------------

for name, p in model.named_parameters():

    if not p.requires_grad:
        continue

    if id(p) not in stiefel_param_ids:
        normal_params.append(p)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(
    "Number of Stiefel parameter tensors:",
    len(stiefel_params)
)

print(
    "Number of individual Stiefel matrices:",
    sum(p.shape[0] for p in stiefel_params)
)

print(
    "Number of normal parameter tensors:",
    len(normal_params)
)

# Verify the groups are disjoint and complete
assert not (
    {id(p) for p in stiefel_params}
    & {id(p) for p in normal_params}
)

assert (
    len(stiefel_params) + len(normal_params)
    == sum(p.requires_grad for p in model.parameters())
)


Stiefel layer: group0.blocks.0.attn.W_q, heads=3, matrix shape=(192, 64)
Stiefel layer: group0.blocks.0.attn.W_k, heads=3, matrix shape=(192, 64)
Stiefel layer: group0.blocks.1.attn.W_q, heads=3, matrix shape=(192, 64)
Stiefel layer: group0.blocks.1.attn.W_k, heads=3, matrix shape=(192, 64)
Stiefel layer: group0.blocks.2.attn.W_q, heads=3, matrix shape=(192, 64)
Stiefel layer: group0.blocks.2.attn.W_k, heads=3, matrix shape=(192, 64)
Stiefel layer: group0.blocks.3.attn.W_q, heads=3, matrix shape=(192, 64)
Stiefel layer: group0.blocks.3.attn.W_k, heads=3, matrix shape=(192, 64)
Stiefel layer: group1.blocks.0.attn.W_q, heads=3, matrix shape=(192, 64)
Stiefel layer: group1.blocks.0.attn.W_k, heads=3, matrix shape=(192, 64)
Stiefel layer: group1.blocks.1.attn.W_q, heads=3, matrix shape=(192, 64)
Stiefel layer: group1.blocks.1.attn.W_k, heads=3, matrix shape=(192, 64)
Stiefel layer: group1.blocks.2.attn.W_q, heads=3, matrix shape=(192, 64)
Stiefel layer: group1.blocks.2.attn.W_k, heads=3, m

In [8]:

# ------------------------------------------------------------
# utils_modify for ViT-Tiny/4
# ------------------------------------------------------------

import math
import torch
import torch.nn as nn


def matrix_norm_one(W, per_head=False):
    """
    Compute the induced matrix 1-norm.

    Supports:
        W: (n, p)       - single matrix
        W: (H, n, p)    - batched attention matrices

    If per_head=False:
        Returns a scalar (maximum over heads).

    If per_head=True:
        Returns the norm for each head.
    """

    if W.ndim not in (2, 3):
        raise ValueError(
            "Expected matrix shape (n,p) or (H,n,p)"
        )

    # Sum absolute values along matrix rows
    column_sums = torch.abs(W).sum(dim=-2)

    # Maximum absolute column sum
    norms = column_sums.amax(dim=-1)

    if per_head:
        return norms

    # Single scalar across all heads
    return norms.amax()


In [9]:

# ------------------------------------------------------------
# gutils_modify for ViT-Tiny/4
# Supports individual (n,p) and batched (H,n,p)
# ------------------------------------------------------------

import torch


def norm(v, dim=-1):
    return v.norm(p=2, dim=dim, keepdim=True)


def unit(v, dim=-1, eps=1e-8):
    vnorm = norm(v, dim)
    return v / vnorm.add(eps), vnorm


def xTy(x, y):
    return torch.sum(x * y, dim=-1, keepdim=True)


def clip_by_norm(v, clip_norm):
    v_norm = norm(v)
    scale = torch.ones_like(v_norm)
    mask = v_norm > clip_norm
    scale[mask] = clip_norm / v_norm[mask]
    return v * scale


def sym_matrix(y):
    return (y + y.transpose(-1, -2)) / 2


def skew_matrix(y):
    return (y - y.transpose(-1, -2)) / 2


# ------------------------------------------------------------
# Stiefel tangent projection
# X, G: (..., n, p), n >= p
# ------------------------------------------------------------

def stiefel_proj_tan(X, G):
    sym = sym_matrix(
        X.transpose(-1, -2) @ G
    )
    return G - X @ sym


# ------------------------------------------------------------
# Stiefel normal projection
# ------------------------------------------------------------

def stiefel_proj_norm(X, G):
    sym = sym_matrix(
        X.transpose(-1, -2) @ G
    )
    return X @ sym


# ------------------------------------------------------------
# Polar retraction
# Input: candidate matrix X + tangent step
# ------------------------------------------------------------

def polar_retraction(Y):
    U, S, Vh = torch.linalg.svd(
        Y, full_matrices=False
    )
    return U @ Vh


# ------------------------------------------------------------
# QR retraction
# ------------------------------------------------------------

def qr_retraction(Y):
    Q, R = torch.linalg.qr(
        Y, mode="reduced"
    )

    d = torch.diagonal(
        R, dim1=-2, dim2=-1
    )

    ph = torch.where(
        d < 0,
        -torch.ones_like(d),
        torch.ones_like(d)
    )

    return Q * ph.unsqueeze(-2)


# ------------------------------------------------------------
# Cayley skew-symmetric generator
# X: (..., n, p)
# U: tangent direction at X
# ------------------------------------------------------------

def cayley_skew(X, U):
    XT = X.transpose(-1, -2)
    UT = U.transpose(-1, -2)

    W = (
        U @ XT
        - X @ UT
        + X @ (UT @ X) @ XT
    )

    return skew_matrix(W)


# ------------------------------------------------------------
# Sequential Cayley iteration
# X: (..., n, p)
# W: (..., n, n), skew-symmetric
# tan_vec = W @ X
# ------------------------------------------------------------

def Cayley_loop(X, W, tan_vec, t, num_iter=5):
    Y = X + t * tan_vec

    for _ in range(num_iter):
        Y = X + 0.5 * t * (
            W @ (X + Y)
        )

    return Y


# ------------------------------------------------------------
# Check Stiefel orthogonality
# ------------------------------------------------------------

def check_identity(X):
    n, p = X.shape[-2:]

    identity = torch.eye(
        p, device=X.device, dtype=X.dtype
    )

    residual = (
        identity - X.transpose(-1, -2) @ X
    )

    errors = torch.linalg.matrix_norm(
        residual, ord="fro"
    )

    print(f"n={n}, p={p}")
    print("Orthogonality errors:", errors)

    return errors


# ------------------------------------------------------------
# Stiefel vector transport by projection
# ------------------------------------------------------------

def stiefel_transport(X, G):
    return stiefel_proj_tan(X, G)


# ------------------------------------------------------------
# Sphere geometry utilities
# Retained from the original implementation
# ------------------------------------------------------------

def gproj(y, g, normalize=False):
    if normalize:
        y, _ = unit(y)

    yTg = xTy(y, g)
    return g - (yTg * y)


def gexp(y, h, normalize=False):
    if normalize:
        y, _ = unit(y)
        h = gproj(y, h)

    u, hnorm = unit(h)
    return y * hnorm.cos() + u * hnorm.sin()


def gpt2(y, h1, h2, normalize=False):
    if normalize:
        h1 = gproj(y, h1)
        h2 = gproj(y, h2)

    u, unorm = unit(h2)
    uTh1 = xTy(u, h1)

    return h1 - uTh1 * (
        unorm.sin() * y
        + (1 - unorm.cos()) * u
    )


def gpt(y, h, normalize=False):
    if normalize:
        h = gproj(y, h)

    u, unorm = unit(h)
    return (
        u * unorm.cos() - y * unorm.sin()
    ) * unorm


In [10]:
X = model.group0.blocks[0].attn.W_q.detach()

# Initial orthogonality
check_identity(X)

# Test tangent projection
G = torch.randn_like(X)
U = stiefel_proj_tan(X, G)

# Descent direction and Cayley generator
direction = -U
W = cayley_skew(X, direction)

# Test the Cayley iteration
Y = Cayley_loop(
    X, W, W @ X,
    t=0.001,
    num_iter=5
)

check_identity(Y)

n=192, p=64
Orthogonality errors: tensor([2.1702e-06, 2.0507e-06, 2.2393e-06])
n=192, p=64
Orthogonality errors: tensor([2.2386e-06, 2.1224e-06, 2.1585e-06])


tensor([2.2386e-06, 2.1224e-06, 2.1585e-06])

In [13]:

# ------------------------------------------------------------
# SGDG Optimizer for ViT-Tiny/4
# ------------------------------------------------------------

import torch
import random
from torch.optim.optimizer import Optimizer, required


class SGDG(Optimizer):
    """
    SGDG optimizer for ViT-Tiny/4.

    Stiefel parameters:
        X: (n, p) or (num_heads, n, p)
        Constraint: X^T X = I

    Normal parameters:
        Standard Euclidean SGD with momentum
        and weight decay.
    """

    def __init__(
        self,
        params,
        lr=required,
        momentum=0,
        dampening=0,
        weight_decay=0,
        nesterov=False,
        stiefel=False,
        omega=0,
        grad_clip=None,
    ):

        defaults = dict(
            lr=lr,
            momentum=momentum,
            dampening=dampening,
            weight_decay=weight_decay,
            nesterov=nesterov,
            stiefel=stiefel,
            omega=omega,
            grad_clip=grad_clip,
        )

        if nesterov and (momentum <= 0 or dampening != 0):
            raise ValueError(
                "Nesterov momentum requires positive "
                "momentum and zero dampening"
            )

        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):

        loss = None

        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:

            lr = group["lr"]
            momentum = group["momentum"]
            weight_decay = group["weight_decay"]
            dampening = group["dampening"]
            nesterov = group["nesterov"]
            stiefel = group["stiefel"]

            for p in group["params"]:

                if p.grad is None:
                    continue

                # --------------------------------------------
                # Stiefel update: Cayley SGDG
                # --------------------------------------------

                if stiefel:

                    X = p.detach()

                    if X.ndim not in (2, 3):
                        raise ValueError(
                            "Stiefel weights must be 2D or 3D"
                        )

                    n, k = X.shape[-2:]

                    if n < k:
                        raise ValueError(
                            "Stiefel requires n >= p"
                        )

                    # Occasional QR correction
                    if random.randint(1, 101) == 1:
                        X = qr_retraction(X)

                    # Euclidean gradient
                    G = p.grad.detach()

                    # Riemannian gradient
                    g = stiefel_proj_tan(X, G)

                    # Momentum buffer
                    state = self.state[p]

                    if "momentum_buffer" not in state:
                        state["momentum_buffer"] = (
                            torch.zeros_like(X)
                        )

                    buffer = state["momentum_buffer"]

                    # Momentum-adjusted tangent direction
                    V = (
                        momentum * stiefel_proj_tan(X, buffer)
                        - g
                    )

                    # Skew-symmetric Cayley generator
                    W = cayley_skew(X, V)

                    # Matrix 1-norm per attention head
                    W_norm = torch.linalg.matrix_norm(
                        W, ord=1
                    )

                    # Adaptive step-size safeguard
                    t = 1.0 / (W_norm + 1e-8)

                    alpha = torch.clamp(
                        t, max=lr
                    )

                    if X.ndim == 3:
                        alpha = alpha[..., None, None]

                    # Sequential Cayley iteration
                    X_new = Cayley_loop(
                        X,
                        W,
                        V,
                        alpha,
                        num_iter=5,
                    )

                    # Update weights
                    p.copy_(X_new)

                    # Transport momentum to new tangent space
                    buffer.copy_(
                        stiefel_proj_tan(X_new, V)
                    )

                # --------------------------------------------
                # Euclidean SGD update
                # --------------------------------------------

                else:

                    d_p = p.grad.detach()

                    if weight_decay != 0:
                        d_p = d_p.add(
                            p, alpha=weight_decay
                        )

                    if momentum != 0:

                        state = self.state[p]

                        if "momentum_buffer" not in state:
                            state["momentum_buffer"] = (
                                d_p.clone()
                            )
                        else:
                            state["momentum_buffer"].mul_(
                                momentum
                            ).add_(
                                d_p,
                                alpha=1 - dampening,
                            )

                        buf = state["momentum_buffer"]

                        if nesterov:
                            d_p = d_p.add(
                                buf, alpha=momentum
                            )
                        else:
                            d_p = buf

                    p.add_(d_p, alpha=-lr)

        return loss


In [14]:

# ------------------------------------------------------------
# ViT-Tiny/4: Optimization hyperparameters
# ------------------------------------------------------------

LR_STIEFEL = 0.2
LR_EUCLID = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005

# ------------------------------------------------------------
# Learning-rate scheduler settings
# ------------------------------------------------------------

LR_MILESTONES = [12, 60, 120, 160]
LR_GAMMA = 0.2

# ------------------------------------------------------------
# Stiefel optimizer: Q and K attention matrices
# ------------------------------------------------------------

stiefel_optimizer = SGDG(
    [
        {
            "params": stiefel_params,
            "lr": LR_STIEFEL,
            "stiefel": True,
            "momentum": 0.0,
            "weight_decay": WEIGHT_DECAY,
        }
    ]
)

# ------------------------------------------------------------
# Euclidean optimizer: remaining ViT parameters
# ------------------------------------------------------------

euclid_optimizer = torch.optim.SGD(
    normal_params,
    lr=LR_EUCLID,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

# ------------------------------------------------------------
# Stiefel learning-rate scheduler
# ------------------------------------------------------------

stiefel_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    stiefel_optimizer,
    milestones=LR_MILESTONES,
    gamma=LR_GAMMA,
)

# ------------------------------------------------------------
# Euclidean learning-rate scheduler
# ------------------------------------------------------------

euclid_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    euclid_optimizer,
    milestones=LR_MILESTONES,
    gamma=LR_GAMMA,
)

print("stiefel_optimizer =", stiefel_optimizer)
print("euclid_optimizer =", euclid_optimizer)


stiefel_optimizer = SGDG (
Parameter Group 0
    dampening: 0
    grad_clip: None
    initial_lr: 0.2
    lr: 0.2
    momentum: 0.0
    nesterov: False
    omega: 0
    stiefel: True
    weight_decay: 0.0005
)
euclid_optimizer = SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    initial_lr: 0.01
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0.0005
)


In [15]:

# ------------------------------------------------------------
# ViT-Tiny/4 Stiefel: Training configuration
# ------------------------------------------------------------

# Set the number of training epochs
num_epochs = 200

print("Starting ViT-Tiny/4 Stiefel training...")
print("Device:", device)
print("Number of epochs:", num_epochs)
print("Batch size:", batch_size)


Starting ViT-Tiny/4 Stiefel training...
Device: cpu
Number of epochs: 200
Batch size: 128


In [ ]:

# ------------------------------------------------------------
# ViT-Tiny/4 Stiefel: Training and validation
# ------------------------------------------------------------

# Training metrics
train_losses = []
train_accuracies = []

# Validation metrics
val_losses = []
val_accuracies = []

# The model was already moved to device during initialization

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    # --------------------------------------------------------
    # Training phase
    # --------------------------------------------------------

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients from both optimizers
        stiefel_optimizer.zero_grad()
        euclid_optimizer.zero_grad()

        # ViT-Tiny/4 forward pass
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Stiefel update for Q and K matrices
        stiefel_optimizer.step()

        # Euclidean update for remaining parameters
        euclid_optimizer.step()

        # Training statistics
        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, dim=1)

        running_correct += (
            predicted == labels
        ).sum().item()

        running_total += labels.size(0)

    # Average training metrics
    epoch_train_loss = running_loss / running_total
    epoch_train_acc = running_correct / running_total

    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_acc)

    # --------------------------------------------------------
    # Validation phase
    # --------------------------------------------------------

    model.eval()

    val_running_loss = 0.0
    val_running_correct = 0
    val_running_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            # ViT-Tiny/4 forward pass
            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += (
                loss.item() * images.size(0)
            )

            _, predicted = torch.max(outputs, dim=1)

            val_running_correct += (
                predicted == labels
            ).sum().item()

            val_running_total += labels.size(0)

    # Average validation metrics
    epoch_val_loss = val_running_loss / val_running_total
    epoch_val_acc = val_running_correct / val_running_total

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)

    # --------------------------------------------------------
    # Learning-rate schedulers
    # --------------------------------------------------------

    # Learning rates used during the current epoch
    current_lr_stiefel = (
        stiefel_optimizer.param_groups[0]["lr"]
    )

    current_lr_euclid = (
        euclid_optimizer.param_groups[0]["lr"]
    )

    # Update schedulers for the next epoch
    stiefel_scheduler.step()
    euclid_scheduler.step()

    # --------------------------------------------------------
    # Print epoch results
    # --------------------------------------------------------

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"LR_Euclid: {current_lr_euclid:.6f} | "
        f"LR_Stiefel: {current_lr_stiefel:.6f} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_acc:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_acc:.4f}"
    )


In [ ]:

# ------------------------------------------------------------
# Check Stiefel constraint errors
# ------------------------------------------------------------

@torch.no_grad()
def compute_stiefel_error(model):

    errors = []

    for module in model.modules():

        if isinstance(module, MultiHeadSelfAttention):

            for W in [module.W_q, module.W_k]:

                p = W.shape[-1]

                I = torch.eye(
                    p,
                    device=W.device,
                    dtype=W.dtype
                )

                residual = (
                    W.transpose(-1, -2) @ W - I
                )

                error = torch.linalg.matrix_norm(
                    residual, ord="fro"
                )

                errors.extend(error.cpu().tolist())

    return max(errors)


print(
    "Maximum Stiefel constraint error:",
    compute_stiefel_error(model)
)


In [ ]:

# ------------------------------------------------------------
# ViT-Tiny/4 Stiefel: Test evaluation
# ------------------------------------------------------------

model.eval()

# Initialize test statistics
test_running_loss = 0.0
test_running_correct = 0
test_running_total = 0

# CIFAR-10 class names
classes = (
    'plane', 'car', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
)

# Per-class statistics
class_correct = {cls_name: 0 for cls_name in classes}
class_total = {cls_name: 0 for cls_name in classes}

# ------------------------------------------------------------
# Test loop
# ------------------------------------------------------------

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # ViT-Tiny/4 forward pass
        outputs = model(images)

        # Cross-entropy loss
        loss = criterion(outputs, labels)

        test_running_loss += (
            loss.item() * images.size(0)
        )

        # Predicted classes
        _, predicted = torch.max(outputs, dim=1)

        # Overall accuracy
        test_running_correct += (
            predicted == labels
        ).sum().item()

        test_running_total += labels.size(0)

        # Per-class statistics
        for true_label, pred_label in zip(labels, predicted):

            true_idx = true_label.item()
            cls_name = classes[true_idx]

            class_total[cls_name] += 1

            if pred_label.item() == true_idx:
                class_correct[cls_name] += 1


# ------------------------------------------------------------
# Overall test metrics
# ------------------------------------------------------------

test_loss = test_running_loss / test_running_total
test_accuracy = test_running_correct / test_running_total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# ------------------------------------------------------------
# Per-class accuracy
# ------------------------------------------------------------

print("\nPer-class accuracy:")

class_accuracies = {}

for cls_name in classes:

    if class_total[cls_name] > 0:

        acc = (
            class_correct[cls_name]
            / class_total[cls_name]
        )

        class_accuracies[cls_name] = acc

        print(f"{cls_name:>5s}: {acc:.4f}")


In [ ]:

# ------------------------------------------------------------
# ViT-Tiny/4 Stiefel: Save training metrics
# ------------------------------------------------------------

import os
import pandas as pd
import torch

output_dir = "vit_stiefel"
os.makedirs(output_dir, exist_ok=True)

results = pd.DataFrame({
    "epoch": list(range(1, len(train_losses) + 1)),
    "train_loss": train_losses,
    "train_accuracy": train_accuracies,
    "val_loss": val_losses,
    "val_accuracy": val_accuracies,
})

metrics_path = os.path.join(
    output_dir, "training_metrics.csv"
)

results.to_csv(metrics_path, index=False)


# ------------------------------------------------------------
# Save ViT-Tiny/4 Stiefel checkpoint
# ------------------------------------------------------------

checkpoint_path = os.path.join(
    output_dir, "vit_tiny4_stiefel_sgdg_checkpoint.pt"
)

torch.save(
    {
        # Architecture
        "model_name": "vit_tiny4",
        "dataset": "CIFAR10",
        "img_size": 32,
        "patch_size": 4,
        "depth": 12,
        "embed_dim": 192,
        "num_heads": 3,
        "num_classes": 10,
        "mlp_ratio": 4.0,
        "dropout": 0.0,

        # Training configuration
        "num_epochs": num_epochs,
        "batch_size": batch_size,
        "lr_stiefel": LR_STIEFEL,
        "lr_euclid": LR_EUCLID,
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "lr_milestones": LR_MILESTONES,
        "lr_gamma": LR_GAMMA,

        # Model state
        "model_state_dict": model.state_dict(),

        # Optimizer states
        "stiefel_optimizer_state_dict":
            stiefel_optimizer.state_dict(),

        "euclid_optimizer_state_dict":
            euclid_optimizer.state_dict(),

        # Scheduler states
        "stiefel_scheduler_state_dict":
            stiefel_scheduler.state_dict(),

        "euclid_scheduler_state_dict":
            euclid_scheduler.state_dict(),

        # Training history
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies,

        # Test results
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "class_accuracies": class_accuracies,
    },
    checkpoint_path,
)

print(f"Saved {metrics_path}")
print(f"Saved {checkpoint_path}")


In [ ]:

# ------------------------------------------------------------
# ViT-Tiny/4 Stiefel: Training plots
# ------------------------------------------------------------

import os
import matplotlib.pyplot as plt

output_dir = "vit_stiefel"
os.makedirs(output_dir, exist_ok=True)

epochs = range(1, len(train_losses) + 1)

# ------------------------------------------------------------
# Training and validation loss
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs, train_losses,
    marker='o', label='Train Loss'
)

plt.plot(
    epochs, val_losses,
    marker='s', label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('ViT-Tiny/4 SGDG: Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.savefig(
    os.path.join(output_dir, "loss_curves.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ------------------------------------------------------------
# Training and validation accuracy (%)
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    [x * 100 for x in train_accuracies],
    marker='o',
    label='Train Accuracy'
)

plt.plot(
    epochs,
    [x * 100 for x in val_accuracies],
    marker='s',
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('ViT-Tiny/4 SGDG: Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.savefig(
    os.path.join(output_dir, "accuracy_curves.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Saved:", os.path.join(output_dir, "loss_curves.png"))
print("Saved:", os.path.join(output_dir, "accuracy_curves.png"))
